In [ ]:
import os
from config import load_env

load_env()  # reads .env: HF_TOKEN, NUM_DOCS, VESPA_MODE, ...

from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    "CC-MAIN-2025-08",
    split="train",
    streaming=True,
)

In [ ]:
from safety import is_adult

NUM_DOCS = int(os.environ["NUM_DOCS"])  # set in .env; the stream is huge, so only take a sample

def to_vespa(doc):
    # FineWeb ids look like '<urn:uuid:d66bc6fe-...>'; strip the wrapper so it is a clean Vespa doc id
    doc_id = doc['id'].removeprefix('<urn:uuid:').removesuffix('>')
    return {
        'id': doc_id,
        'fields': {
            'id': doc_id,
            'url': doc['url'],
            'text': doc['text'],
            'dump': doc['dump'],
            'year': int(doc['date'][:4]),  # date is e.g. '2013-05-18T05:48:54Z'
            'token_count': doc['token_count'],
            'adult': is_adult(doc['text'], doc['url']),
        }
    }

def feed_stream():
    # Generator: documents are converted and fed one by one, never all held in memory
    for doc in dataset.take(NUM_DOCS):
        yield to_vespa(doc)

print(NUM_DOCS, to_vespa(next(iter(dataset.take(1))))['fields']['url'])

In [ ]:
from vespa.application import ApplicationPackage

app_package = ApplicationPackage(name='fineweb')

In [ ]:
from vespa.package import Schema, Document

film_schema = Schema(name='fineweb_schema', document=Document())

In [ ]:
from vespa.package import Field

film_schema.add_fields(
    Field(
        name='url',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='text',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='dump',
        type='string',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='year',
        type='int',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='token_count',
        type='int',
        indexing=['attribute', 'summary']
    ),
    Field(
        name='id',
        type='string',
        indexing=['summary']
    ),
    Field(
        name='adult',  # set by safety.is_adult(); SafeSearch filters on it
        type='bool',
        indexing=['attribute', 'summary'],
        attribute=['fast-search']
    )
)

In [ ]:
from vespa.package import FieldSet, RankProfile

film_schema.add_field_set(
    FieldSet(
        name='default',
        fields=['url', 'text']
    ))

# Rank on both fields, boosting url matches a bit
film_schema.add_rank_profile(
    RankProfile(
        name='default',
        first_phase='bm25(text) + 2 * bm25(url)'
    ))

In [ ]:
app_package.add_schema(film_schema)

In [ ]:
# Run once to create the app. After a restart skip this cell and run the connect cell below.
if os.environ.get("VESPA_MODE", "local") == "local":
    from vespa.deployment import VespaDocker

    vespa_docker = VespaDocker(
        port=int(os.environ["VESPA_LOCAL_PORT"]),
        container_memory=int(float(os.environ["VESPA_CONTAINER_MEMORY_GB"]) * 1024**3),
        # named volumes keep the index across container restarts/removal
        volumes=["vespa-var:/opt/vespa/var", "vespa-logs:/opt/vespa/logs"],
    )
    vespa_docker.deploy(application_package=app_package)
else:
    from vespa.deployment import VespaCloud

    vespa_cloud = VespaCloud(
        tenant='akashsinghal',
        application='fineweb',
        application_package=app_package
    )
    vespa_cloud.deploy(region="aws-euw1-az1")

In [ ]:
from config import connect_vespa

# Works after a deploy or a kernel restart; local/cloud is picked from VESPA_MODE in .env
vespa_application = connect_vespa()
print(os.environ.get("VESPA_MODE", "local"), vespa_application.url, vespa_application.port)

In [ ]:
import time
from vespa.io import VespaResponse

def callback(response: VespaResponse, id: str):
    if not response.is_successful():
        print(f"Error when feeding document {id}: {response.get_json()}")

# Feeding is idempotent (same id overwrites), so it is safe to re-run if interrupted
t0 = time.time()
vespa_application.feed_iterable(
    feed_stream(),
    schema='fineweb_schema',
    callback=callback,
    max_workers=8,
    max_queue_size=1000,
    max_connections=8,
)
print(f"Fed {NUM_DOCS} docs in {(time.time() - t0) / 60:.1f} min")

In [ ]:
from vespa.io import VespaQueryResponse

def print_hits(response: VespaQueryResponse):
    for i, hit in enumerate(response.hits):
        f = hit['fields']
        print(f'{i+1:>3}  {f["id"][:8]}  {f["year"]}  {f["token_count"]:>6} tok  {f["url"]}')

In [ ]:
# Searching for 'tornado' in the default fieldset (url + text)

response = vespa_application.query(
    yql="select * from sources * where text contains 'tornado'", # The default limit in Vespa is 10 results
)

print_hits(response)

In [ ]:
# Using userQuery(): the query text is passed as a separate parameter

response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 100",
    query="tornado damage",
)

print_hits(response)

In [ ]:
# Search by text: matches words in the url or the body
response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 10",
    query="tornado damage",
)

print_hits(response)

In [ ]:
# Search by url: pass a full url or a fragment; punctuation is tokenized away
response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 10",
    query="http://ap.org/Content/Press-Release/2012/How-AP-reported-in-all-formats-from-tornado-stricken-regions",
    type="any",  # match any term, ranked by how many match
)

print_hits(response)

In [ ]:
# Restrict to one field when you only want url or only text
response = vespa_application.query(
    yql="select * from sources * where url contains 'ap' and text contains 'tornado' limit 10",
)

print_hits(response)